In [1]:
# Automatically reload .py files when they are changed
%load_ext autoreload
%autoreload 2

# Numerical packages
import numpy as np
from scipy import optimize

# Plotting
import matplotlib.pyplot as plt

plt.rcParams.update({
    'axes.grid': True,
    'grid.color': 'black',
    'grid.alpha': 0.25,
    'grid.linestyle': '-'
})
plt.rcParams.update({'font.size': 14})

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

# Import the model classes
from Consumer import ConsumerClass
from Government import GovernmentClass

print("Everything is set up correctly!")

Everything is set up correctly!


# 1. A consumer with two nests

I consider a consumer who chooses between food, bus trips and train trips. Bus and train belong to a lower CES nest and are first combined into a travel composite. I then combine this travel composite with food in an upper CES nest.


## 1.1 The problem in nested budget shares

Instead of choosing quantities directly, I formulate the problem in nested budget shares. I let $s_1$ denote the share of income spent on food and $w$ denote the share of the remaining travel budget spent on bus trips.

The resulting budget shares are

$$
s_1,\qquad
s_2=(1-s_1)w,\qquad
s_3=(1-s_1)(1-w).
$$

I therefore always have

$$
s_1+s_2+s_3=1.
$$

This formulation ensures that the budget constraint is satisfied automatically. I can therefore solve the problem over the simple box $[0,1]\times[0,1]$.


## 1.2 Calibration

I consider two calibrations. I keep all parameters fixed except for the elasticity of substitution between bus and train, $\sigma_B$. In the complements calibration I set $\sigma_B=0.40$, while in the substitutes calibration I set $\sigma_B=3.00$.

In [8]:
# a. I create the complements calibration
model_complements = ConsumerClass()

# b. I create the substitutes calibration
model_substitutes = ConsumerClass(
    par={'sigma_B': 3.0}
)

# c. I print both calibrations to check the parameters
print(model_complements)

print('')

print(model_substitutes)

ConsumerClass
  alpha = 0.6000, beta = 0.5000
  sigma_A = 0.8000, sigma_B = 0.4000
  p1 = 1.0000, p2 = 1.0000, p3 = 1.5000
  I = 10.0000

ConsumerClass
  alpha = 0.6000, beta = 0.5000
  sigma_A = 0.8000, sigma_B = 3.0000
  p1 = 1.0000, p2 = 1.0000, p3 = 1.5000
  I = 10.0000


## 1.3 How do you know your answer is right?

### 1.3.1 Is the answer possible?

Since there is no closed-form solution for this model, I first check whether the numerical solution satisfies the basic requirements of the consumer problem. I check that all three budget shares are strictly between zero and one, that they sum to one, and that all three quantities are positive.

In [14]:
def check_solution(model,opt):
    """ Check whether a numerical solution satisfies the basic model requirements. """

    # a. I collect the three budget shares
    shares = np.array([
        opt.s1,
        opt.s2,
        opt.s3
    ])

    # b. I check that all budget shares are strictly between zero and one
    assert np.all(
        shares > 0.0
    ), 'at least one budget share is not positive'

    assert np.all(
        shares < 1.0
    ), 'at least one budget share is not below one'

    # c. I check that the three budget shares sum to one
    assert np.isclose(
        np.sum(shares),
        1.0
    ), 'budget shares do not sum to one'

    # d. I translate the optimal nested shares into quantities
    x1,x2,x3 = model.quantities(
        opt.s1,
        opt.w
    )

    quantities = np.array([
        x1,
        x2,
        x3
    ])

    # e. I check that all optimal quantities are positive
    assert np.all(
        quantities > 0.0
    ), 'at least one optimal quantity is not positive'

    return True

### 1.3.2 Do two different methods agree?

As a second check, I solve the same consumer problem using two different numerical methods: a two-dimensional grid search and L-BFGS-B. I compare the solutions in Section 2 after implementing both methods. If the two methods produce approximately the same optimal nested budget shares and utility, this provides additional evidence that the numerical solution is correct.

In [15]:
def check_methods_agree(model,N=1000,atol=0.002):
    """ Check whether grid search and L-BFGS-B give approximately the same solution. """

    # a. I solve the model using a fine grid search
    opt_grid = model.solve_grid(
        N=N,
        do_print=False
    )

    # b. I solve the same model using L-BFGS-B
    opt_lbfgsb = model.solve(
        do_print=False
    )

    # c. I collect the two nested budget shares from each method
    grid_solution = np.array([
        opt_grid.s1,
        opt_grid.w
    ])

    lbfgsb_solution = np.array([
        opt_lbfgsb.s1,
        opt_lbfgsb.w
    ])

    # d. I calculate the absolute difference between the two solutions
    difference = np.abs(
        grid_solution - lbfgsb_solution
    )

    # e. I check that the two numerical methods give approximately the same answer
    assert np.allclose(
        grid_solution,
        lbfgsb_solution,
        atol=atol
    ), 'grid search and L-BFGS-B do not give approximately the same solution'

    # f. I return the solutions so I can inspect the comparison
    return {
        'grid_s1': opt_grid.s1,
        'grid_w': opt_grid.w,
        'L-BFGS-B_s1': opt_lbfgsb.s1,
        'L-BFGS-B_w': opt_lbfgsb.w,
        'difference_s1': difference[0],
        'difference_w': difference[1]
    }

In [16]:
# a. I check the complements calibration
check_complements = check_methods_agree(
    model_complements
)

# b. I check the substitutes calibration
check_substitutes = check_methods_agree(
    model_substitutes
)

print('Complements:')
print(check_complements)

print('')

print('Substitutes:')
print(check_substitutes)

print('')
print('Both numerical checks passed')

Complements:
{'grid_s1': np.float64(0.5355355355355356), 'grid_w': np.float64(0.4394394394394394), 'L-BFGS-B_s1': np.float64(0.5356233806953101), 'L-BFGS-B_w': np.float64(0.43947902810370126), 'difference_s1': np.float64(8.784515977455776e-05), 'difference_w': np.float64(3.958866426184704e-05)}

Substitutes:
{'grid_s1': np.float64(0.5385385385385385), 'grid_w': np.float64(0.6926926926926927), 'L-BFGS-B_s1': np.float64(0.5382253431946789), 'L-BFGS-B_w': np.float64(0.6923077683280843), 'difference_s1': np.float64(0.0003131953438596513), 'difference_w': np.float64(0.00038492436460846324)}

Both numerical checks passed


## 2. Solving the consumer problem numerically


### 2.1 Two-dimensional grid search

I first solve the consumer problem using a two-dimensional grid search over the nested budget shares $s_1$ and $w$. I compare different grid sizes to examine how the numerical solution changes as I make the grid finer and how the number of utility evaluations increases. I perform the analysis for both calibrations.

In [10]:
import pandas as pd


def compare_grid_sizes(model,N_values):
    """ I solve the model for several grid sizes and compare the solutions. """

    # a. I create an empty list in which I store the results
    results = []

    # b. I use this variable to remember the previous grid solution
    previous_solution = None

    # c. I solve the model for each grid size
    for N in N_values:

        # i. I solve the consumer problem using an N x N grid
        opt_grid = model.solve_grid(
            N=N,
            do_print=False
        )

        # ii. I check that the numerical solution is feasible
        check_solution(
            model,
            opt_grid
        )

        # iii. I calculate how much the solution moves relative
        # to the previous grid size
        if previous_solution is None:

            movement = np.nan

        else:

            movement = np.sqrt(
                (opt_grid.s1 - previous_solution[0])**2
                +
                (opt_grid.w - previous_solution[1])**2
            )

        # iv. I store the results from this grid size
        results.append({
            'N': N,
            's1': opt_grid.s1,
            'w': opt_grid.w,
            's2': opt_grid.s2,
            's3': opt_grid.s3,
            'utility': opt_grid.u,
            'movement_from_previous': movement,
            'function_evaluations': opt_grid.nfev
        })

        # v. I save the current solution so I can compare it
        # with the next, finer grid
        previous_solution = (
            opt_grid.s1,
            opt_grid.w
        )

    # d. I collect the results in a DataFrame
    return pd.DataFrame(
        results
    )


# e. I choose the grid sizes requested in the assignment
N_values = [
    50,
    100,
    500,
    1000
]


# f. I run the grid comparison for the complements calibration
grid_complements = compare_grid_sizes(
    model_complements,
    N_values
)


# g. I run the same grid comparison for the substitutes calibration
grid_substitutes = compare_grid_sizes(
    model_substitutes,
    N_values
)

In [11]:
grid_complements

,N,s1,w,s2,s3,utility,movement_from_previous,function_evaluations
0,50,0.530612,0.448980,0.210746,0.258642,3.400748,NaN,2500
1,100,0.535354,0.444444,0.206510,0.258137,3.401482,0.006561,10000
2,500,0.535070,0.438878,0.204047,0.260882,3.401674,0.005574,250000
3,1000,0.535536,0.439439,0.204104,0.260360,3.401680,0.000729,1000000


In [12]:
grid_substitutes

,N,s1,w,s2,s3,utility,movement_from_previous,function_evaluations
0,50,0.530612,0.693878,0.325698,0.143690,3.484594,NaN,2500
1,100,0.535354,0.696970,0.323845,0.140802,3.485005,0.005660,10000
2,500,0.539078,0.691383,0.318673,0.142248,3.485097,0.006715,250000
3,1000,0.538539,0.692693,0.319651,0.141810,3.485104,0.001417,1000000


The grid search becomes more precise as I increase the number of grid points, but the solution does not improve completely smoothly. For example, the movement in the substitutes calibration is slightly larger when I increase the grid from \(N=100\) to \(N=500\) than when I increase it from \(N=50\) to \(N=100\). This happens because the grid search can only choose among the discrete points contained in each grid.

The computational cost increases rapidly. Since I use \(N\) values for each of the two choice variables, I evaluate utility \(N^2\) times. Increasing the grid from \(N=100\) to \(N=1000\) therefore increases the number of utility evaluations from 10,000 to 1,000,000. I obtain a more accurate solution, but at a substantially higher computational cost.